# 06 - Current trends: web research plus the product catalogue

**Scenario.** Product marketing asks how **2026 TV display trends** (OLED, mini-LED, high refresh rates) compare with
what our **Lumenix OLED O65** offers. Internal manuals cannot know current trends, so the Coordinator adds the
**Research agent** (Serper web search) alongside the Product Manuals agent.

**Prerequisites:** `.env` with `OPENAI_API_KEY`, `JUDGE_OPENAI_API_KEY` and `SERPER_API_KEY`; the vector stores (`make ingest`); the CRM & Support MCP server (`make mcp-server-up`).

In [1]:
# Setup: make src/ importable and load the settings from .env
import sys, warnings
from pathlib import Path

ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
sys.path.insert(0, str(ROOT / "src"))
warnings.filterwarnings("ignore", message=".*langchain.mcp.*")

from agentic_rag.config import AgentSettings

settings = AgentSettings.load(ROOT)
print(f"Coordinator: {settings.model} ({settings.coordinator_effort}) | specialists: {settings.specialist_model} "
      f"({settings.specialist_effort}) | judge: {settings.judge_model} (pass >= {settings.judge_pass_score})")

Coordinator: gpt-5.1 (high) | specialists: gpt-5.1 (low) | judge: gpt-5.5 (pass >= 80)


## Serper directly

In [2]:
from agentic_rag.web import serper_search

for r in serper_search("TV display technology trends 2026 OLED mini-LED", settings.serper_api_key, num=5):
    print(f"- {r.title} ({r.date or 'no date'})\n  {r.url}")

- 2026 TV Lineup: The Year Of RGB Mini LED (6 days ago)
  https://www.rtings.com/tv/learn/2026-lineup
- OLED vs Mini-LED in 2026 (3 months ago)
  https://www.youtube.com/watch?v=k9pwg8_V7kc
- Mini LED vs OLED: Which TV Display Technology Is Better ... (4 days ago)
  https://www.reliancedigital.in/c/resource-center/bg/mini-led-vs-oled
- New TV Technology Coming in 2026 (21 Jan 2026)
  https://www.consumerreports.org/electronics-computers/tvs/tv-trends-to-watch-for-a9692962731/
- Can The Best LED TV Finally Challenge OLED? New 2026 TV ... (2 weeks ago)
  https://www.youtube.com/watch?v=-Ke0W4T8BlM


In [3]:
# Start the multi-agent service: SQLite memory + CRM MCP tools + 5 specialists + Coordinator graph
from agentic_rag.graph import CoordinatorService

service = await CoordinatorService.create(settings)
print(service.crm.describe())
print(service.roster())

CRM MCP server at http://localhost:8765/mcp: customer=['get_customer_profile', 'search_customer_profiles'], ticket_read=['get_tickets_by_customer', 'search_tickets'], ticket_write=['register_ticket', 'resolve_ticket']
- `hr_policy` (available): HR policies (leave, travel and expenses, medical insurance, conduct, POSH, WFH, IT security, privacy, attendance, appraisal, whistleblowing, anti-bribery, notice period). Current versions by default; flags superseded values.
- `product_manuals` (available): TV and phone user manuals: specifications, setup, error codes, troubleshooting steps, warranty and safety. Pins the exact model (near-duplicate manuals).
- `customer_support` (available): Find customers by ID, name, email or phone and report their profile: active status, credit limit, contact details, remarks such as products owned. Read-only.
- `ticket_management` (available): Support ticket history and search (by customer, status, priority, product model, text). Proposes new tickets or reso

In [4]:
QUESTION = ("What are the main TV display technology trends in 2026 (OLED, mini-LED, refresh rates, gaming features), "
            "and how does our Lumenix OLED O65 compare on those points?")

result = await service.ask(QUESTION, user_id="marketing-nisha", on_event=print)
result

[   0.0s] start_turn      done
[   0.0s] load_memory     1 long-term facts recalled
[  14.3s] plan            Initial plan for this question. -> s1=product_manuals, s2=research
[  36.1s] run_step        s1 product_manuals done in 21.8s (confidence 0.86, 5 evidence, 0 proposed actions)
[  71.6s] run_step        s2 research done in 57.3s (confidence 0.79, 15 evidence, 0 proposed actions)
[ 121.4s] reflect         synthesize: The two specialist reports collectively cover both the Lumenix OLED O65’s key specs and the broader 2025–2026 TV technol
[ 164.9s] synthesize      done
[ 169.7s] judge           attempt 1: 86.3/100 PASS (ground 76, rel 95, compl 90, ctx 88)
[ 169.7s] finalize        done
[ 173.4s] save_memory     done


**status** `completed` | **thread** `thread-db214ed3` | **elapsed** 173.4s  
**judge** 86.3/100 | **passed** True | **attempts** 1 | **specialist runs** 2

```text
plan round 1 (Initial plan for this question.): s1:product_manuals, s2:research
timeline:
    s1 product_manuals       0.0s ->   21.8s |###############                         |
    s2 research              0.0s ->   57.3s |########################################|
judge attempt 1: 86.3/100 PASS (ground 76, rel 95, compl 90, ctx 88)
```

---

Here’s the short version:

- 2026 premium TVs:  
  - OLED: Brighter, with new QD‑OLED and MLA panels at the high end. Still prized for perfect blacks; burn‑in is managed but not eliminated. [E6][E7][E8][E9][E10]  
  - Mini‑LED: Thousands of dimming zones and very high peak brightness, especially for bright rooms; often brighter than most OLEDs. [E11][E12][E13][E14]  
  - Refresh rates: 4K/120 Hz is now the baseline for “serious” gaming TVs; 4K/144–165 Hz (and even higher at 1080p) is the PC‑enthusiast step‑up. [E15][E16][E17][E18]  
  - Gaming features: Multiple HDMI 2.1 ports with 4K/120, VRR, ALLM and low input lag are expected; top sets add 4K/144, FreeSync/G‑Sync, and Dolby Vision gaming. [E15][E16][E18][E19][E20]

- Lumenix OLED O65:  
  - Hits the 2026 “must‑have” list: 4K OLED, native 120 Hz, 4× HDMI 2.1 all with 4K/120, VRR, ALLM, Game Mode and Game Bar. [E1][E2][E3][E4][E5]  
  - Manual doesn’t state peak brightness, color gamut, panel generation (QD‑OLED vs MLA‑WOLED), >120 Hz modes, Dolby Vision gaming support, or exact input‑lag numbers, so we can’t fully benchmark it against the very brightest or most PC‑focused 2026 TVs.  

Below is a bit more detail by topic.

---

### 1. OLED trends vs. Lumenix OLED O65

**2026 OLED trends**

- Modern OLEDs are noticeably brighter and more vivid than earlier generations while retaining perfect blacks and contrast. [E6]  
- High‑end lines increasingly use:
  - **QD‑OLED**: Quantum dots for color conversion, with consistently wider color gamut and higher brightness than classic white‑OLED (WOLED). [E8]  
  - **MLA‑enhanced WOLED**: Micro Lens Array layers that improve brightness extraction from existing emitters, narrowing the gap with QD‑OLED. [E9]  
- Panel makers are demoing extremely bright QD‑OLED panels (e.g., a 77" panel up to ~4,500‑nit peak prior to CES 2026), though retail models run lower to protect image quality and lifespan. [E7]  
- Burn‑in is treated as a real but manageable risk, mitigated through pixel‑shifting, logo dimming and panel refresh cycles across brands. [E10]

**Where the O65 fits**

- The O65 is a 65" 4K self‑lit OLED TV with a native 120 Hz panel. [E1]  
- It supports HDR10, HLG, and Dolby Vision IQ for tone‑mapped HDR playback. [E1]  
- The manual **does not specify**:
  - Whether the panel is QD‑OLED or an MLA‑enhanced WOLED.  
  - Peak SDR/HDR brightness in nits.  
  - Color gamut coverage (e.g., DCI‑P3 or Rec.2020 percentages).  
- Because of these gaps, we can say it clearly belongs in the modern OLED category, but we **cannot confirm** if it reaches the very top tier of 2026 OLED brightness or color volume, nor exactly how it compares to QD‑OLED/MLA flagships.

Implication: For dark‑room or controlled‑light viewing, the O65’s OLED blacks and Dolby Vision IQ position it well. For very bright rooms, some 2026 QD‑OLED or MLA‑OLED competitors may be brighter, but we lack hard brightness numbers for a precise comparison.  

---

### 2. Mini‑LED trends (context for positioning)

**2026 mini‑LED landscape**

- Upper‑midrange and flagship mini‑LED TVs at 65–75" commonly use **thousands of local dimming zones**, with models like Hisense’s U8K reaching up to 5,120 zones. [E12][E13]  
- They are marketed as ideal for bright rooms, with micro‑lens arrays and dense backlights pushing peak luminance while avoiding OLED burn‑in risks. [E11]  
- Reviews and user commentary consistently note that only a small subset of OLEDs can reach the highest HDR peaks of the best mini‑LEDs. [E14]

**O65 vs mini‑LED trend**

- As an OLED, the O65 competes on:
  - Perfect blacks and pixel‑level contrast. [Implicit in OLED category; see E6 on OLED contrast]  
  - Cinematic HDR formats (Dolby Vision IQ, HDR10, HLG). [E1]  
- Mini‑LED rivals compete on:
  - Much higher peak brightness and bright‑room punch. [E11][E14]  
- Without published nits for the O65, we can’t state how close it gets to those brightest mini‑LEDs, but by technology nature and market commentary, mini‑LED still leads in raw brightness while OLED (including the O65) leads in black levels and uniformity.

---

### 3. Refresh‑rate and motion trends vs. O65

**2026 refresh‑rate standards**

- Buying guides state that the best gaming TVs now typically use **120 Hz panels**, with some stretching to **144–165 Hz** refresh rates, particularly for PC gaming. [E17]  
- High‑end models can support **4K @ 144 Hz and 1080p at very high frame rates (up to 288 Hz)** for PC users. [E15]  
- Flagship QLED/mini‑LED sets advertise refresh rates up to 144 Hz along with full HDMI 2.1 gaming feature support. [E18]

**O65’s motion capabilities**

- The O65 has a **native 120 Hz** OLED panel. [E1]  
- All four HDMI ports are HDMI 2.1 and support **4K at 120 Hz**. [E1][E2]  
- The manual does **not** list any overclocked or PC‑specific modes above 120 Hz (no 144 Hz/165 Hz spec).  
- Motion controls:
  - “Motion Smoothing” with Off/Low/Medium/High settings for interpolation. [E5]  
  - Filmmaker Mode disables motion smoothing for accurate film playback; Game Mode disables most processing for responsiveness. [E3]

Positioning: The O65 meets the 2026 **premium/gaming baseline** of 4K/120 on a native 120 Hz panel, but it doesn’t advertise the >120 Hz modes that differentiate the most PC‑centric gaming TVs. [E15][E16][E17][E18]

---

### 4. Gaming‑feature stack vs. O65

**2026 gaming expectations**

- Guidance for PS5/Xbox/PC buyers is to prioritise:  
  - **HDMI 2.1**,  
  - **VRR (Variable Refresh Rate)**,  
  - **ALLM (Auto Low Latency Mode)**,  
  - plus a native **120 Hz** panel. [E19]  
- Flagship sets add:
  - Multiple HDMI 2.1 ports,  
  - VRR with often AMD FreeSync and/or Nvidia G‑Sync, [E16][E18][E20]  
  - Support for 4K 144 Hz where possible, [E15][E18][E20]  
  - Dolby Vision gaming at 4K/120 on compatible platforms, [E20]  
  - Very low input lag (typically < ~10 ms in game mode). [E15]

**O65’s gaming package**

- Connectivity:
  - **Four HDMI 2.1 ports**, all supporting 4K at 120 Hz. [E1][E2]  
  - HDMI 2 supports eARC for soundbars/AVRs. [E1][E2]  
- Gaming modes and controls:
  - A dedicated **Game picture mode** that “minimises input lag by skipping most picture processing.” [E3][E4]  
  - **VRR and ALLM** supported up to 120 Hz on HDMI 2.1 ports when using an Ultra High Speed HDMI cable. [E4]  
  - **Game Optimiser / Game Bar**: lets you toggle VRR, ALLM, “Input Lag Boost”, and see current frame rate, HDR status, and input‑lag mode while playing. [E4][E5]  
- HDR for games:
  - Supports HDR10 and Dolby Vision IQ (commonly leveraged for games on Xbox/PC when implemented), plus HLG. [E1]

What’s missing or not documented:

- The manual **does not specify**:
  - Whether VRR includes AMD FreeSync and/or Nvidia G‑Sync compatibility, whereas some 2026 gaming TVs explicitly advertise both. [E16][E20]  
  - Whether **Dolby Vision gaming at 4K/120** is supported; it only mentions Dolby Vision IQ in general. [E1][E20]  
  - Exact **input‑lag figures in ms**. It simply describes low input lag and an Input Lag Boost setting. [E4][E5]  
  - Any support for 4K/144 Hz or higher‑refresh 1080p modes. [E15][E16]

Positioning:  
- Versus 2026 expectations, the O65 is **fully aligned on the core stack** (4K/120, HDMI 2.1 on all ports, VRR, ALLM, dedicated Game Mode and Game Bar). [E1][E2][E4][E5][E19]  
- It may lag top gaming flagships on **specialised PC features** (4K/144–165 Hz, branded FreeSync/G‑Sync, explicit Dolby Vision gaming support, and published sub‑10 ms lag), but those are step‑up differentiators rather than table‑stakes. [E15][E16][E18][E20]

---

### Summary for product positioning

- **Strengths relative to 2026 trends**
  - Modern 4K OLED with native 120 Hz and premium HDR formats. [E1]  
  - Full‑fat gaming connectivity: 4× HDMI 2.1, 4K/120, VRR, ALLM, Game Mode, Game Bar. [E1][E2][E4][E5]  
  - Strong home‑cinema positioning thanks to OLED contrast and Filmmaker Mode. [E1][E3]

- **Where it’s likely behind the 2026 cutting edge**
  - No documented use of QD‑OLED/MLA or peak‑brightness metrics, so it may not match the brightest OLED or mini‑LED flagships in very bright rooms. [E6][E7][E8][E9][E11][E14]  
  - No advertised >120 Hz modes, explicit FreeSync/G‑Sync, or measured sub‑10 ms lag—features highlighted by the most PC/gaming‑oriented 2026 TVs. [E15][E16][E17][E18][E20]

- **Unknowns (cannot be verified from current documentation)**
  - Panel generation (QD‑OLED vs MLA‑WOLED), peak nits, color gamut percentages.  
  - Exact input lag at 4K/60 and 4K/120.  
  - Formal support for Dolby Vision gaming at 120 Hz and specific VRR standards like FreeSync/G‑Sync.  

If you share who you see as your main competitive set (specific brands/models), we can map the O65 more explicitly against those on gaming and picture‑quality points.

In [ ]:
# Web evidence carries URLs and dates; manual evidence carries document IDs and pages
for e in result.final.get("evidence", []):
    print(f"{e['id']:>4} {e['kind']:<4} {e['ref'][:80]} {e.get('date') or ''}")

In [ ]:
# Close the memory databases
await service.aclose()

## What to observe
- The Research agent is used only for external, time-sensitive facts; specs come from the manual.
- Web evidence lists URL and date; the answer separates market trends from product facts.